## Partie 4 : Préparation Power BI — Export, DAX & Schéma Étoile

> **Objectif** : Exporter toutes les tables du schéma en étoile et les agrégats nécessaires au dashboard Power BI, en s'assurant de la qualité des clés de relation.

---
**Auteur** : Malcom Closse — Marketing Data Analyst  
**Prérequis** : Parties 1, 2 & 3 exécutées  
**Stack** : Python · Pandas · PostgrSQL

---
## Import & connexion

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os

TABLES = os.path.join("..", "outputs", "tables")
PBI = os.path.join("..", "outputs", "powerbi")

user = "postgres"  # ou ton utilisateur défini dans le docker-compose
password = "postgres"  # idem
host = "localhost"
port = "5432"
database = "retailstore_db"

# Création de l'engine SQLAlchemy
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{database}")


def sql(q):
    # Avec SQLAlchemy, on passe directement l'objet engine à Pandas
    return pd.read_sql_query(q, engine)


print(" Connexion PostgreSQL OK")

# Vérification : Liste des tables dans Postgres
q_list_tables = """
SELECT table_name 
FROM information_schema.tables 
WHERE table_schema = 'public';
"""
print(" Tables disponibles :", sql(q_list_tables)["table_name"].tolist())

 Connexion PostgreSQL OK
   Tables disponibles : ['factsales', 'dimdate', 'dimstore', 'dimdept', 'dimmarkdown']


---
## Export des 5 tables du schéma étoile

In [3]:
star_tables = ["factsales", "dimdate", "dimstore", "dimdept", "dimmarkdown"]

print("EXPORT SCHÉMA EN ÉTOILE")
for name in star_tables:
    df = sql(f"SELECT * FROM {name}")
    path = os.path.join(PBI, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"  {name:<18} {len(df):>10,} lignes × {df.shape[1]:>2} col.")

EXPORT SCHÉMA EN ÉTOILE
  factsales             156,000 lignes ×  5 col.
  dimdate                   156 lignes ×  8 col.
  dimstore                   50 lignes ×  5 col.
  dimdept                    20 lignes ×  2 col.
  dimmarkdown             7,800 lignes × 10 col.


---
## Tables agrégées pour les visuels Power BI

In [ ]:
# KPI globaux
kpi = """
SELECT
    COUNT(*) AS total_obs,
    COUNT(DISTINCT storeid)  AS nb_stores,
    COUNT(DISTINCT deptid) AS nb_depts,
    ROUND(SUM(weeklysalesclean)::numeric, 0) AS ca_total,
    ROUND(AVG(weeklysalesclean)::numeric, 2) AS ca_moyen,
    ROUND(MAX(weeklysalesclean)::numeric, 0) AS ca_max,
    SUM(isholiday) AS nb_sem_feriees, 
    ROUND(AVG(isholiday)::numeric * 100, 1) AS pct_ferie       -- Plus besoin de ::int ici non plus
FROM factsales;
"""
kpi = sql(kpi)
kpi.to_csv(os.path.join(PBI, "AGG_kpi_globaux.csv"), index=False)
print(" AGG_kpi_globaux.csv")
display(kpi)


 AGG_kpi_globaux.csv


,total_obs,nb_stores,nb_depts,ca_total,ca_moyen,ca_max,nb_sem_feriees,pct_ferie
0,156000,50,20,8.814544e+09,56503.49,505959.0,18000.0,11.5


In [ ]:
# ── CA par type × size ──────────────────────────────────────
ca_type = """
SELECT
    s.storetype,
    s.sizecategory,
    s.region,
    COUNT(DISTINCT f.storeid) AS nb_stores,
    ROUND(SUM(f.weeklysalesclean)::numeric, 0) AS ca_total,
    ROUND(AVG(f.weeklysalesclean)::numeric, 2) AS ca_moyen
FROM factsales f
JOIN dimstore s ON f.storeid = s.storeid
GROUP BY s.storetype, s.sizecategory, s.region
ORDER BY ca_total DESC;
"""
ca_type = sql(ca_type)
ca_type.to_csv(os.path.join(PBI, "AGG_ca_type_size.csv"), index=False)
print(" AGG_ca_type_size.csv")

 AGG_ca_type_size.csv


In [22]:
# ── CA par mois ─────────────────────────────────────────────
ca_month = """
SELECT
    d.year,
    d.month,
    d.monthname,
    d.season,
    ROUND(SUM(f.weeklysalesclean)::numeric, 0) AS ca_total,
    ROUND(AVG(f.weeklysalesclean)::numeric, 2) AS ca_moyen,
    COUNT(*) AS nb_obs
FROM factsales f
JOIN dimdate d ON f.date = d.date
GROUP BY d.year, d.month, d.monthname, d.season
ORDER BY d.year, d.month
"""
ca_month = sql(ca_month)
ca_month.to_csv(os.path.join(PBI, "AGG_ca_mensuel.csv"), index=False)
print(" AGG_ca_mensuel.csv")

 AGG_ca_mensuel.csv


In [24]:
# ── Impact holiday ──────────────────────────────────────────
# Exploite holiday_name et season — colonnes spécifiques à ton dataset
holiday = """
SELECT
    d.holidaylabel,
    d.holidayname,
    d.season,
    s.storetype,
    COUNT(*) AS nb_obs,
    ROUND(AVG(f.weeklysalesclean)::numeric, 2) AS ca_moyen,
    ROUND(SUM(f.weeklysalesclean)::numeric, 0) AS ca_total
FROM factsales f
JOIN dimdate  d ON f.date = d.date
JOIN dimstore s ON f.storeid = s.storeid
GROUP BY d.holidaylabel, d.holidayname, d.season, s.storetype
ORDER BY d.holidaylabel, s.storetype
"""
holiday = sql(holiday)
holiday.to_csv(os.path.join(PBI, "AGG_impact_holiday.csv"), index=False)
print(" AGG_impact_holiday.csv")

 AGG_impact_holiday.csv


In [26]:
# ── Impact markdown ─────────────────────────────────────────
markdown = """
SELECT
    CASE
        WHEN COALESCE(m.totalmarkdown, 0) = 0  THEN '0_Aucune'
        WHEN m.totalmarkdown < 5000 THEN '1_Faible'
        WHEN m.totalmarkdown < 20000 THEN '2_Moderee'
        WHEN m.totalmarkdown < 50000 THEN '3_Forte'
        ELSE '4_Tres_forte'
    END AS segment_promo,
    s.storetype,
    s.region,
    COUNT(*) AS nb_obs,
    ROUND(AVG(f.weeklysalesclean)::numeric, 2) AS ca_moyen,
    ROUND(AVG(COALESCE(m.totalmarkdown, 0))::numeric, 2)    AS md_moyen
FROM factsales f
JOIN dimstore s ON f.storeid = s.storeid
LEFT JOIN dimmarkdown m ON f.storeid = m.storeid AND f.date = m.date
GROUP BY segment_promo, s.storetype, s.region
ORDER BY segment_promo, s.storetype
"""
markdown = sql(markdown)
markdown.to_csv(os.path.join(PBI, "AGG_impact_markdown.csv"), index=False)
print(" AGG_impact_markdown.csv")

 AGG_impact_markdown.csv


In [30]:
# ── Performance stores ──────────────────────────────────────
stores_agg = """
SELECT
    f.storeid,
    s.storelabel,
    s.storetype,
    s.storesize,
    s.sizecategory,
    s.region,
    ROUND(SUM(f.weeklysalesclean)::numeric, 0)  AS ca_total,
    ROUND(AVG(f.weeklysalesclean)::numeric, 2)  AS ca_moyen,
    COUNT(DISTINCT f.date) AS nb_semaines,
    COUNT(DISTINCT f.deptid) AS nb_depts,
    RANK()  OVER (ORDER BY AVG(f.weeklysalesclean) DESC) AS rang,
    NTILE(3) OVER (ORDER BY AVG(f.weeklysalesclean) DESC) AS tercile
FROM factsales f
JOIN dimstore s ON f.storeid = s.storeid
GROUP BY 1,2,3,4,5,6
ORDER BY ca_total DESC
"""
stores_agg = sql(stores_agg)
stores_agg.to_csv(os.path.join(PBI, "AGG_stores_performance.csv"), index=False)
print(" AGG_stores_performance.csv")

 AGG_stores_performance.csv


In [31]:
# ── Top depts ───────────────────────────────────────────────
depts_agg = """
SELECT
    f.deptid,
    d.deptlabel,
    ROUND(SUM(f.weeklysalesclean)::numeric, 0) AS ca_total,
    ROUND(AVG(f.weeklysalesclean)::numeric, 2) AS ca_moyen,
    COUNT(DISTINCT f.storeid) AS nb_stores
FROM factsales f
JOIN dimdept d ON f.deptid = d.deptid
GROUP BY f.deptid, d.deptlabel -- Correction ici : ajout de d.deptlabel
ORDER BY ca_total DESC;
"""
depts_agg = sql(depts_agg)
depts_agg.to_csv(os.path.join(PBI, "AGG_depts_performance.csv"), index=False)
print(" AGG_depts_performance.csv")

# Nettoyage de la connexion SQLAlchemy en fin de script
engine.dispose()
print(
    f"\n Masterclass SQL terminée ! Tous les exports Power BI sont disponibles dans → {PBI}"
)

 AGG_depts_performance.csv

 Masterclass SQL terminée ! Tous les exports Power BI sont disponibles dans → ../outputs/powerbi


---
## Validation du schéma étoile — Intégrité référentielle

In [33]:
fact = pd.read_csv(os.path.join(PBI, "factsales.csv"))
dim_s = pd.read_csv(os.path.join(PBI, "dimstore.csv"))
dim_d = pd.read_csv(os.path.join(PBI, "dimdate.csv"))
dim_dp = pd.read_csv(os.path.join(PBI, "dimdept.csv"))
dim_md = pd.read_csv(os.path.join(PBI, "dimmarkdown.csv"))

checks = [
    ("storeid fact → dimstore", set(fact["storeid"]) - set(dim_s["storeid"])),
    ("deptid  fact → dimdept", set(fact["deptid"]) - set(dim_dp["deptid"])),
    ("date  fact → dimdate", set(fact["date"]) - set(dim_d["date"])),
]

print("VALIDATION CLÉS ÉTRANGÈRES ")
all_ok = True
for label, orphans in checks:
    status = "OK" if not orphans else f" {len(orphans)} orphelins"
    print(f"  {label:<35} {status}")
    if orphans:
        all_ok = False

print(f"\n  Lignes FACT_SALES   : {len(fact):>10,}")
print(f"  Lignes DIM_DATE     : {len(dim_d):>10,}")
print(f"  Lignes DIM_STORE    : {len(dim_s):>10,}")
print(f"  Lignes DIM_DEPT     : {len(dim_dp):>10,}")
print(f"  Lignes DIM_MARKDOWN : {len(dim_md):>10,}")
print(
    f"\n  Schéma étoile : {' Prêt pour Power BI' if all_ok else ' Corrections nécessaires'}"
)

VALIDATION CLÉS ÉTRANGÈRES 
  storeid fact → dimstore             OK
  deptid  fact → dimdept              OK
  date  fact → dimdate                OK

  Lignes FACT_SALES   :    156,000
  Lignes DIM_DATE     :        156
  Lignes DIM_STORE    :         50
  Lignes DIM_DEPT     :         20
  Lignes DIM_MARKDOWN :      7,800

  Schéma étoile :  Prêt pour Power BI


---
## Récap final

In [34]:
all_files = sorted(os.listdir(PBI))
print("  PARTIE 4 TERMINÉE — Données prêtes pour Power BI")
print(f"\n  {len(all_files)} fichiers dans outputs/powerbi/ :")
for f in all_files:
    size = os.path.getsize(os.path.join(PBI, f))
    print(f"     {f:<40} {size / 1024:>6.1f} KB")
print("\n   Ordre import Power BI :")
steps = [
    "1. Importer les 5 tables du schéma étoile (FACT + 4 DIM)",
    "2. Créer les relations dans la vue Modèle",
    "3. Importer les tables AGG_ pour les visuels pré-agrégés",
    "4. Créer les mesures DAX (voir DAX_Retail_PowerBI.md)",
    "5. Construire les visuels selon le guide de mise en page",
]
for s in steps:
    print(f"     {s}")

  PARTIE 4 TERMINÉE — Données prêtes pour Power BI

  12 fichiers dans outputs/powerbi/ :
     AGG_ca_mensuel.csv                          1.7 KB
     AGG_ca_type_size.csv                        0.7 KB
     AGG_depts_performance.csv                   0.7 KB
     AGG_impact_holiday.csv                      1.6 KB
     AGG_impact_markdown.csv                     1.4 KB
     AGG_kpi_globaux.csv                         0.1 KB
     AGG_stores_performance.csv                  3.3 KB
     dimdate.csv                                 7.4 KB
     dimdept.csv                                 0.2 KB
     dimmarkdown.csv                           619.6 KB
     dimstore.csv                                1.4 KB
     factsales.csv                            4165.8 KB

   Ordre import Power BI :
     1. Importer les 5 tables du schéma étoile (FACT + 4 DIM)
     2. Créer les relations dans la vue Modèle
     3. Importer les tables AGG_ pour les visuels pré-agrégés
     4. Créer les mesures DAX (voir DAX